# LightRet â€” Stage 1: BERT â†’ RetBERT Sentence Distillation

**Goal**: Teach RetBERT (word-level, RetVec backbone) to produce sentence embeddings
that match frozen BERT-Base. Both models process clean text.  
**Loss**: `1 âˆ’ cosine_similarity(z_RetBERT, z_BERT)` averaged over the batch.

## Required Kaggle Datasets (add before running)
| Dataset name | Contents |
|---|---|
| `lightret-source` | Upload the entire `lightret/` project folder (src/, train_*.py, etc.) |
| `lightret-weights` | Upload `retvec_v1_weights.npz` |

> **Internet**: must be **ON** â€” HuggingFace downloads BERT weights, WikiText-103, CoNLL-2003.

## Expected runtime (T4 GPU)
- Dataset loading: ~5 min (WikiText-103 is ~500 MB)
- Per epoch: ~90â€“120 min (WikiText sentences dominate)
- Total 5 epochs: ~8â€“10 hours  
  â†’ Consider reducing `STAGE1_EPOCHS` to 2â€“3 for a first run.

## Output
`/kaggle/working/weights/retbert_stage1.pt` â€” download and use as input for Stage 2.

In [ ]:
# â”€â”€ 1. Install packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# torch, numpy are pre-installed on Kaggle
!pip install -q transformers datasets

In [ ]:
# â”€â”€ 2. Setup working directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Copies the project source and weights into /kaggle/working/ so that
# `import src.*` works and config.py ROOT resolves correctly.

import os, shutil, pathlib

# !! Adjust these if your Kaggle dataset names differ !!
SOURCE_DATASET  = '/kaggle/input/lightret-source'   # contains src/, train_*.py
WEIGHTS_DATASET = '/kaggle/input/lightret-weights'  # contains retvec_v1_weights.npz

WORK = pathlib.Path('/kaggle/working')

# Copy src/ to working dir
src_dst = WORK / 'src'
if src_dst.exists():
    shutil.rmtree(src_dst)
shutil.copytree(f'{SOURCE_DATASET}/src', str(src_dst))

# Copy RetVec weights
shutil.copy(f'{WEIGHTS_DATASET}/retvec_v1_weights.npz', str(WORK / 'retvec_v1_weights.npz'))

# Create weights output directory
(WORK / 'weights').mkdir(exist_ok=True)

os.chdir(str(WORK))
print('Working directory:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))


# -- Write latest dataset.py from embedded source (no GitHub needed) --
import base64 as _b64
_D = (
    "IiIiCmRhdGEvZGF0YXNldC5weSDigJQgRGF0YXNldCBjbGFzc2VzIGFuZCBjb2xsYXRpb24gZm9yIGFs"
    "bCB0aHJlZSB0cmFpbmluZyBzdGFnZXMuCgpQcmV0cmFpbkRhdGFzZXQgIChTdGFnZSAxICYgMikKICAg"
    "IENvbWJpbmVzIFdpa2lUZXh0LTEwMy1yYXcgKyBDb05MTC0yMDAzIHRyYWluIGludG8gYSBmbGF0IGxp"
    "c3Qgb2YKICAgIHdvcmQtdG9rZW5pemVkIHNlbnRlbmNlcy4gUmV0dXJucyBsaXN0W3N0cl0gcGVyIGl0"
    "ZW0uCgpORVJEYXRhc2V0ICAgICAgIChTdGFnZSAzKQogICAgTG9hZHMgQ29OTEwtMjAwMy4gT24gZWFj"
    "aCBfX2dldGl0ZW1fXywgYXBwbGllcyBzdG9jaGFzdGljIG5vaXNlIHRvCiAgICB0aGUgY2xlYW4gc2Vu"
    "dGVuY2UsIHByb2plY3RzIEJJTyBsYWJlbHMgdG8gbm9pc3kgY29vcmRpbmF0ZXMsIGFuZAogICAgcmV0"
    "dXJucyBib3RoIHJlcHJlc2VudGF0aW9ucyBwbHVzIHRoZSB3b3JkLWxldmVsIGFsaWdubWVudC4KCkNv"
    "bGxhdGlvbjoKICAgIHByZXRyYWluX2NvbGxhdGUg4oCUIGlkZW50aXR5IChtb2RlbHMgYWNjZXB0IGxp"
    "c3RbbGlzdFtzdHJdXSkKICAgIG5lcl9jb2xsYXRlICAgICAg4oCUIHBhZHMgbGFiZWwgdGVuc29ycywg"
    "YnVuZGxlcyBpbnRvIGEgZGljdAoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMK"
    "CmltcG9ydCByZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDb05M"
    "TC0yMDAzIGxvYWRlciAgKHNjcmlwdC1mcmVlLCB3b3JrcyB3aXRoIGRhdGFzZXRzID49IDMuMCkKIyAt"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0KCmRlZiBfbG9hZF9jb25sbDIwMDMoc3BsaXQ6IHN0cik6CiAgICAiIiIKICAg"
    "IExvYWQgQ29OTEwtMjAwMyBmcm9tIGxvY2FsIGRpc2sgZmlyc3QsIHRoZW4gZmFsbCBiYWNrIHRvIEh1"
    "Z2dpbmdGYWNlIEh1Yi4KCiAgICBkYXRhc2V0cyA+PSAzLjAgcmVmdXNlcyB0byBydW4gZGF0YXNldCBz"
    "Y3JpcHRzIChjb25sbDIwMDMucHkpLgogICAgUHJlZmVycmVkIHBhdGg6IGxvYWQgZnJvbSBwcmUtc2F2"
    "ZWQgQXJyb3cgZGF0YXNldCB1cGxvYWRlZCBhcyBhIEthZ2dsZSBkYXRhc2V0LgogICAgRmFsbGJhY2s6"
    "IGxvYWQgcGFycXVldCBmaWxlcyBkaXJlY3RseSB2aWEgaGY6Ly8gVVJJIChieXBhc3NlcyBzY3JpcHQg"
    "ZGV0ZWN0aW9uKS4KCiAgICBPbiBLYWdnbGU6IGFkZCBjb25sbDIwMDNfbG9jYWwgYXMgYSBkYXRhc2V0"
    "IGlucHV0LCBpdCBtb3VudHMgYXQKICAgIC9rYWdnbGUvaW5wdXQvY29ubGwyMDAzLWxvY2FsL2Nvbmxs"
    "MjAwM19sb2NhbC8KICAgICIiIgogICAgaW1wb3J0IHBhdGhsaWIKICAgIGZyb20gZGF0YXNldHMgaW1w"
    "b3J0IGxvYWRfZnJvbV9kaXNrLCBsb2FkX2RhdGFzZXQKCiAgICAjIFN0cmF0ZWd5IDE6IGxvY2FsIHBy"
    "ZS1zYXZlZCBkYXRhc2V0IChLYWdnbGUgZGF0YXNldCBpbnB1dCkKICAgIGxvY2FsX2NhbmRpZGF0ZXMg"
    "PSBbCiAgICAgICAgcGF0aGxpYi5QYXRoKCIva2FnZ2xlL2lucHV0L2NvbmxsMjAwMy1sb2NhbC9jb25s"
    "bDIwMDNfbG9jYWwiKSwKICAgICAgICBwYXRobGliLlBhdGgoIi9rYWdnbGUvaW5wdXQvZGF0YXNldHMv"
    "bWFoZXNoYmFkYXJpL2xpZ2h0cmV0LXNvdXJjZS9saWdodHJldC9jb25sbDIwMDNfbG9jYWwvY29ubGwy"
    "MDAzX2xvY2FsIiksCiAgICAgICAgcGF0aGxpYi5QYXRoKCJjb25sbDIwMDNfbG9jYWwiKSwgICMgbG9j"
    "YWwgZGV2CiAgICBdCiAgICBwcmludChmIltjb25sbDIwMDNdIGxvYWRpbmcgc3BsaXQ9J3tzcGxpdH0n"
    "IikKICAgIGZvciBsb2NhbF9wYXRoIGluIGxvY2FsX2NhbmRpZGF0ZXM6CiAgICAgICAgc3BsaXRfcGF0"
    "aCA9IGxvY2FsX3BhdGggLyBzcGxpdAogICAgICAgIHByaW50KGYiICB0cnlpbmcgbG9jYWw6IHtzcGxp"
    "dF9wYXRofSAtPiAiLCBlbmQ9IiIsIGZsdXNoPVRydWUpCiAgICAgICAgaWYgc3BsaXRfcGF0aC5leGlz"
    "dHMoKToKICAgICAgICAgICAgcHJpbnQoIkZPVU5EIikKICAgICAgICAgICAgcmV0dXJuIGxvYWRfZnJv"
    "bV9kaXNrKHN0cihzcGxpdF9wYXRoKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludCgibm90"
    "IGZvdW5kIikKCiAgICAjIFN0cmF0ZWd5IDI6IGRpcmVjdCBwYXJxdWV0IHZpYSBoZjovLyBVUkkgKG5v"
    "IHNjcmlwdCBpbnZvbHZlZCkKICAgIGJhc2UgPSAiaGY6Ly9kYXRhc2V0cy9jb25sbDIwMDMvZGF0YSIK"
    "ICAgIHBhcnF1ZXRfdXJsID0gZiJ7YmFzZX0ve3NwbGl0fS0wMDAwMC1vZi0wMDAwMS5wYXJxdWV0Igog"
    "ICAgcHJpbnQoZiIgIHRyeWluZyBwYXJxdWV0OiB7cGFycXVldF91cmx9IC0+ICIsIGVuZD0iIiwgZmx1"
    "c2g9VHJ1ZSkKICAgIHRyeToKICAgICAgICBkcyA9IGxvYWRfZGF0YXNldCgicGFycXVldCIsIGRhdGFf"
    "ZmlsZXM9e3NwbGl0OiBwYXJxdWV0X3VybH0sIHNwbGl0PXNwbGl0KQogICAgICAgIHByaW50KCJPSyIp"
    "CiAgICAgICAgcmV0dXJuIGRzCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQo"
    "ZiJGQUlMRUQgKHtlfSkiKQoKICAgICMgU3RyYXRlZ3kgMzogZHluYW1pY2FsbHkgZGlzY292ZXIgcGFy"
    "cXVldCBmaWxlcyBpbiB0aGUgcmVwbwogICAgcHJpbnQoIiAgdHJ5aW5nIGR5bmFtaWMgcGFycXVldCBk"
    "aXNjb3ZlcnkgLT4gIiwgZW5kPSIiLCBmbHVzaD1UcnVlKQogICAgdHJ5OgogICAgICAgIGZyb20gaHVn"
    "Z2luZ2ZhY2VfaHViIGltcG9ydCBsaXN0X3JlcG9fZmlsZXMKICAgICAgICBmaWxlcyA9IFsKICAgICAg"
    "ICAgICAgZiJoZjovL2RhdGFzZXRzL2NvbmxsMjAwMy97Zn0iCiAgICAgICAgICAgIGZvciBmIGluIGxp"
    "c3RfcmVwb19maWxlcygiY29ubGwyMDAzIiwgcmVwb190eXBlPSJkYXRhc2V0IikKICAgICAgICAgICAg"
    "aWYgZi5lbmRzd2l0aCgiLnBhcnF1ZXQiKSBhbmQgZiIve3NwbGl0fS0iIGluIGYKICAgICAgICBdCiAg"
    "ICAgICAgaWYgZmlsZXM6CiAgICAgICAgICAgIGRzID0gbG9hZF9kYXRhc2V0KCJwYXJxdWV0IiwgZGF0"
    "YV9maWxlcz17c3BsaXQ6IGZpbGVzfSwgc3BsaXQ9c3BsaXQpCiAgICAgICAgICAgIHByaW50KGYiT0sg"
    "KHtsZW4oZmlsZXMpfSBmaWxlcykiKQogICAgICAgICAgICByZXR1cm4gZHMKICAgICAgICBlbHNlOgog"
    "ICAgICAgICAgICBwcmludCgibm8gcGFycXVldCBmaWxlcyBmb3VuZCIpCiAgICBleGNlcHQgRXhjZXB0"
    "aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiJGQUlMRUQgKHtlfSkiKQoKICAgICMgU3RyYXRlZ3kgNDog"
    "bGFzdCByZXNvcnQgKG9sZGVyIGRhdGFzZXRzIHZlcnNpb25zKQogICAgcHJpbnQoIiAgdHJ5aW5nIHRy"
    "dXN0X3JlbW90ZV9jb2RlIC0+ICIsIGVuZD0iIiwgZmx1c2g9VHJ1ZSkKICAgIHRyeToKICAgICAgICBk"
    "cyA9IGxvYWRfZGF0YXNldCgiY29ubGwyMDAzIiwgc3BsaXQ9c3BsaXQsIHRydXN0X3JlbW90ZV9jb2Rl"
    "PVRydWUpCiAgICAgICAgcHJpbnQoIk9LIikKICAgICAgICByZXR1cm4gZHMKICAgIGV4Y2VwdCBFeGNl"
    "cHRpb24gYXMgZToKICAgICAgICBwcmludChmIkZBSUxFRCAoe2V9KSIpCiAgICAgICAgcmFpc2UgUnVu"
    "dGltZUVycm9yKAogICAgICAgICAgICBmIkFsbCBzdHJhdGVnaWVzIGZhaWxlZCBmb3IgQ29OTEwtMjAw"
    "MyBzcGxpdD0ne3NwbGl0fScuICIKICAgICAgICAgICAgIlVwbG9hZCBjb25sbDIwMDNfbG9jYWwvIGFz"
    "IGEgS2FnZ2xlIGRhdGFzZXQgYW5kIGFkZCBpdCBhcyBpbnB1dC4iCiAgICAgICAgKQoKaW1wb3J0IHRv"
    "cmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldAoKZnJvbSBzcmMuY29uZmlnIGlt"
    "cG9ydCAoCiAgICBORVJfSUdOT1JFX0lOREVYLAogICAgU1RBR0UxX01BWF9XT1JEUywKICAgIFNUQUdF"
    "Ml9NQVhfV09SRFMsCiAgICBTVEFHRTNfTUFYX1dPUkRTLAogICAgU1RBR0UxX1dJS0lURVhUX0RBVEFT"
    "RVQsCiAgICBTVEFHRTFfV0lLSVRFWFRfQ09ORklHLAogICAgU1RBR0UxX0NPTkxMX0RBVEFTRVQsCiAg"
    "ICBTVEFHRTNfQ09OTExfREFUQVNFVCwKICAgIE5PSVNFX1BfU1VCLAogICAgTk9JU0VfUF9JTlMsCiAg"
    "ICBOT0lTRV9QX0RFTCwKICAgIE5PSVNFX1BfU1BBQ0VfSU5TLAopCmZyb20gc3JjLm5vaXNlIGltcG9y"
    "dCBhcHBseV9ub2lzZSwgYnVpbGRfd29yZF9hbGlnbm1lbnQKZnJvbSBzcmMuZGF0YS5sYWJlbF91dGls"
    "cyBpbXBvcnQgcHJvamVjdF9sYWJlbHMKCl9TRU5UX1NQTElUX1JFID0gcmUuY29tcGlsZShyJyg/PD1b"
    "LiE/XSlccysnKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgSGVscGVycwojIC0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoK"
    "ZGVmIF9zZW50ZW5jZXNfZnJvbV93aWtpdGV4dCh0ZXh0OiBzdHIsIG1heF93b3JkczogaW50KSAtPiBs"
    "aXN0W2xpc3Rbc3RyXV06CiAgICAiIiIKICAgIEV4dHJhY3Qgd29yZC10b2tlbml6ZWQgc2VudGVuY2Vz"
    "IGZyb20gYSBXaWtpVGV4dC0xMDMgcmF3IHRleHQgY2h1bmsuCgogICAgRmlsdGVyaW5nOgogICAgLSBT"
    "a2lwIGVtcHR5IGxpbmVzCiAgICAtIFNraXAgc2VjdGlvbiBoZWFkaW5ncyAobGluZXMgc3RhcnRpbmcg"
    "d2l0aCAnPScpCiAgICAtIFNwbGl0IGVhY2ggbGluZSBpbnRvIHNlbnRlbmNlcyBhdCBbLiE/XSBib3Vu"
    "ZGFyaWVzCiAgICAtIEtlZXAgc2VudGVuY2VzIHdpdGggPj0gNCB3b3JkczsgdHJ1bmNhdGUgdG8gbWF4"
    "X3dvcmRzCiAgICAiIiIKICAgIHNlbnRlbmNlczogbGlzdFtsaXN0W3N0cl1dID0gW10KICAgIGZvciBs"
    "aW5lIGluIHRleHQuc3BsaXQoIlxuIik6CiAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAg"
    "IGlmIG5vdCBsaW5lIG9yIGxpbmUuc3RhcnRzd2l0aCgiPSIpOgogICAgICAgICAgICBjb250aW51ZQog"
    "ICAgICAgIGZvciBwYXJ0IGluIF9TRU5UX1NQTElUX1JFLnNwbGl0KGxpbmUpOgogICAgICAgICAgICB3"
    "b3JkcyA9IHBhcnQuc3BsaXQoKQogICAgICAgICAgICBpZiBsZW4od29yZHMpID49IDQ6CiAgICAgICAg"
    "ICAgICAgICBzZW50ZW5jZXMuYXBwZW5kKHdvcmRzWzptYXhfd29yZHNdKQogICAgcmV0dXJuIHNlbnRl"
    "bmNlcwoKCmRlZiBfcGFkX2xhYmVsX2JhdGNoKGxhYmVsX2xpc3RzOiBsaXN0W2xpc3RbaW50XV0pIC0+"
    "IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgUGFkIGEgbGlzdCBvZiBsYWJlbC1pZCBsaXN0cyB3aXRo"
    "IE5FUl9JR05PUkVfSU5ERVguCiAgICBSZXR1cm5zIChCLCBMX21heCkgTG9uZ1RlbnNvci4KICAgICIi"
    "IgogICAgbWF4X2xlbiA9IG1heChsZW4obGJsKSBmb3IgbGJsIGluIGxhYmVsX2xpc3RzKQogICAgcGFk"
    "ZGVkID0gdG9yY2guZnVsbCgobGVuKGxhYmVsX2xpc3RzKSwgbWF4X2xlbiksIE5FUl9JR05PUkVfSU5E"
    "RVgsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBmb3IgaSwgbGJsIGluIGVudW1lcmF0ZShsYWJlbF9saXN0"
    "cyk6CiAgICAgICAgcGFkZGVkW2ksIDogbGVuKGxibCldID0gdG9yY2gudGVuc29yKGxibCwgZHR5cGU9"
    "dG9yY2gubG9uZykKICAgIHJldHVybiBwYWRkZWQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFByZXRyYWlu"
    "RGF0YXNldAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgUHJldHJhaW5EYXRhc2V0KERhdGFzZXQpOgog"
    "ICAgIiIiCiAgICBXb3JkLXRva2VuaXplZCBzZW50ZW5jZXMgZm9yIFN0YWdlIDEgKEJFUlTihpJSZXRC"
    "RVJUKSBhbmQgU3RhZ2UgMgogICAgKFJldEJFUlTihpJMaWdodFJldCkgZGlzdGlsbGF0aW9uLgoKICAg"
    "IFNvdXJjZXM6CiAgICAgICAgV2lraVRleHQtMTAzLXJhdy12MSAg4oCUIGJyb2FkIHZvY2FidWxhcnkg"
    "Y292ZXJhZ2UKICAgICAgICBDb05MTC0yMDAzIHRyYWluICAgICDigJQgZG9tYWluIG92ZXJsYXAgd2l0"
    "aCBTdGFnZSAzIE5FUiBkYXRhCgogICAgQXJnczoKICAgICAgICBzcGxpdCAgICAgOiBIdWdnaW5nRmFj"
    "ZSBzcGxpdCBuYW1lICgndHJhaW4nIG9yICd2YWxpZGF0aW9uJykKICAgICAgICBtYXhfd29yZHMgOiB0"
    "cnVuY2F0aW9uIGxlbmd0aCAoZGVmYXVsdCBTVEFHRTFfTUFYX1dPUkRTID0gNjQpCiAgICAgICAgdmVy"
    "Ym9zZSAgIDogcHJpbnQgbG9hZGluZyBwcm9ncmVzcwogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAog"
    "ICAgICAgIHNlbGYsCiAgICAgICAgc3BsaXQ6IHN0ciA9ICJ0cmFpbiIsCiAgICAgICAgbWF4X3dvcmRz"
    "OiBpbnQgPSBTVEFHRTFfTUFYX1dPUkRTLAogICAgICAgIHZlcmJvc2U6IGJvb2wgPSBUcnVlLAogICAg"
    "KSAtPiBOb25lOgogICAgICAgIGZyb20gZGF0YXNldHMgaW1wb3J0IGxvYWRfZGF0YXNldCAgIyBpbXBv"
    "cnRlZCBoZXJlIHRvIGtlZXAgdG9wLWxldmVsIGltcG9ydCBsaWdodAoKICAgICAgICBzZWxmLnNlbnRl"
    "bmNlczogbGlzdFtsaXN0W3N0cl1dID0gW10KCiAgICAgICAgIyAtLS0tIFdpa2lUZXh0LTEwMyAtLS0t"
    "CiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoIkxvYWRpbmcgV2lraVRleHQtMTAz"
    "IC4uLiIpCiAgICAgICAgd2lraSA9IGxvYWRfZGF0YXNldCgKICAgICAgICAgICAgU1RBR0UxX1dJS0lU"
    "RVhUX0RBVEFTRVQsCiAgICAgICAgICAgIFNUQUdFMV9XSUtJVEVYVF9DT05GSUcsCiAgICAgICAgICAg"
    "IHNwbGl0PXNwbGl0LAogICAgICAgICkKICAgICAgICBmb3IgaXRlbSBpbiB3aWtpOgogICAgICAgICAg"
    "ICBzZWxmLnNlbnRlbmNlcy5leHRlbmQoX3NlbnRlbmNlc19mcm9tX3dpa2l0ZXh0KGl0ZW1bInRleHQi"
    "XSwgbWF4X3dvcmRzKSkKCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoZiIgIFdp"
    "a2lUZXh0IHNlbnRlbmNlcyA6IHtsZW4oc2VsZi5zZW50ZW5jZXMpOix9IikKCiAgICAgICAgIyAtLS0t"
    "IENvTkxMLTIwMDMgLS0tLQogICAgICAgIGlmIHZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KCJMb2Fk"
    "aW5nIENvTkxMLTIwMDMgLi4uIikKICAgICAgICBjb25sbF9zcGxpdCA9ICJ0cmFpbiIgaWYgc3BsaXQg"
    "PT0gInRyYWluIiBlbHNlICJ2YWxpZGF0aW9uIgogICAgICAgIGNvbmxsID0gX2xvYWRfY29ubGwyMDAz"
    "KGNvbmxsX3NwbGl0KQogICAgICAgIG5fYmVmb3JlID0gbGVuKHNlbGYuc2VudGVuY2VzKQogICAgICAg"
    "IGZvciBpdGVtIGluIGNvbmxsOgogICAgICAgICAgICB3b3JkcyA9IGl0ZW1bInRva2VucyJdCiAgICAg"
    "ICAgICAgIGlmIGxlbih3b3JkcykgPj0gMjoKICAgICAgICAgICAgICAgIHNlbGYuc2VudGVuY2VzLmFw"
    "cGVuZCh3b3Jkc1s6bWF4X3dvcmRzXSkKCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJp"
    "bnQoZiIgIENvTkxMIHNlbnRlbmNlcyAgICA6IHtsZW4oc2VsZi5zZW50ZW5jZXMpIC0gbl9iZWZvcmU6"
    "LH0iKQogICAgICAgICAgICBwcmludChmIiAgVG90YWwgICAgICAgICAgICAgIDoge2xlbihzZWxmLnNl"
    "bnRlbmNlcyk6LH0iKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4g"
    "bGVuKHNlbGYuc2VudGVuY2VzKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHg6IGludCkgLT4g"
    "bGlzdFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLnNlbnRlbmNlc1tpZHhdCgoKIyAtLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0KIyBORVJEYXRhc2V0CiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpjbGFzcyBORVJEYXRhc2V0KERhdGFz"
    "ZXQpOgogICAgIiIiCiAgICBDb05MTC0yMDAzIE5FUiBkYXRhc2V0IHdpdGggb24tdGhlLWZseSBub2lz"
    "ZSBhdWdtZW50YXRpb24gZm9yIFN0YWdlIDMuCgogICAgRWFjaCBfX2dldGl0ZW1fXyBjYWxsIGFwcGxp"
    "ZXMgZnJlc2ggc3RvY2hhc3RpYyBub2lzZSB0byB0aGUgY2xlYW4gc2VudGVuY2UsCiAgICBwcm9kdWNp"
    "bmcgZGlmZmVyZW50IG5vaXNlIHBhdHRlcm5zIGFjcm9zcyBlcG9jaHMgYXV0b21hdGljYWxseS4KCiAg"
    "ICBSZXR1cm5zIGEgZGljdCB3aXRoOgogICAgICAgIGNsZWFuX3dvcmRzICA6IGxpc3Rbc3RyXSAgICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgIGNsZWFuIHdvcmQgdG9rZW5zCiAgICAgICAgY2xlYW5fbGFi"
    "ZWxzIDogbGlzdFtpbnRdICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgQklPIGxhYmVsIElEcyAo"
    "Y2xlYW4pCiAgICAgICAgbm9pc3lfd29yZHMgIDogbGlzdFtzdHJdICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgd29yZCB0b2tlbnMgYWZ0ZXIgbm9pc2UKICAgICAgICBub2lzeV9sYWJlbHMgOiBsaXN0"
    "W2ludF0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICBCSU8gbGFiZWwgSURzIChub2lzeSBjb29y"
    "ZHMpCiAgICAgICAgYWxpZ25tZW50ICAgIDogbGlzdFt0dXBsZVtsaXN0W2ludF0sIGxpc3RbaW50XV1d"
    "ICAgICAgKENfaywgTl9rKSBncm91cHMKCiAgICBBcmdzOgogICAgICAgIHNwbGl0ICAgICAgOiAndHJh"
    "aW4nIHwgJ3ZhbGlkYXRpb24nIHwgJ3Rlc3QnCiAgICAgICAgbWF4X3dvcmRzICA6IHRydW5jYXRlIGNs"
    "ZWFuIHNlbnRlbmNlIHRvIHRoaXMgbWFueSB3b3JkcwogICAgICAgIG5vaXNlX3Byb2IgOiBpZiBGYWxz"
    "ZSwgcmV0dXJucyBjbGVhbj09bm9pc3kgKGZvciBpbmZlcmVuY2UgLyBldmFsdWF0aW9uKQogICAgIiIi"
    "CgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgc3BsaXQ6IHN0ciA9ICJ0cmFp"
    "biIsCiAgICAgICAgbWF4X3dvcmRzOiBpbnQgPSBTVEFHRTNfTUFYX1dPUkRTLAogICAgICAgIGFwcGx5"
    "X25vaXNlX2F1ZzogYm9vbCA9IFRydWUsCiAgICApIC0+IE5vbmU6CiAgICAgICAgZnJvbSBkYXRhc2V0"
    "cyBpbXBvcnQgbG9hZF9kYXRhc2V0CgogICAgICAgIHNlbGYubWF4X3dvcmRzICAgICAgID0gbWF4X3dv"
    "cmRzCiAgICAgICAgc2VsZi5hcHBseV9ub2lzZV9hdWcgPSBhcHBseV9ub2lzZV9hdWcKCiAgICAgICAg"
    "ZGF0YXNldCA9IF9sb2FkX2NvbmxsMjAwMyhzcGxpdCkKCiAgICAgICAgIyBuZXJfdGFncyBtYXkgYmUg"
    "aW50IChDbGFzc0xhYmVsKSBvciBzdHIgZGVwZW5kaW5nIG9uIHRoZSBkYXRhc2V0IHNvdXJjZQogICAg"
    "ICAgIGZyb20gc3JjLmNvbmZpZyBpbXBvcnQgTkVSX0xBQkVMMklECiAgICAgICAgZGVmIF90b19pbnQo"
    "dGFnKSAtPiBpbnQ6CiAgICAgICAgICAgIHJldHVybiB0YWcgaWYgaXNpbnN0YW5jZSh0YWcsIGludCkg"
    "ZWxzZSBORVJfTEFCRUwySURbdGFnXQoKICAgICAgICBzZWxmLl9kYXRhOiBsaXN0W3R1cGxlW2xpc3Rb"
    "c3RyXSwgbGlzdFtpbnRdXV0gPSBbXQogICAgICAgIGZvciBpdGVtIGluIGRhdGFzZXQ6CiAgICAgICAg"
    "ICAgIHdvcmRzICA9IGl0ZW1bInRva2VucyJdWzogbWF4X3dvcmRzXQogICAgICAgICAgICBsYWJlbHMg"
    "PSBbX3RvX2ludCh0KSBmb3IgdCBpbiBpdGVtWyJuZXJfdGFncyJdWzogbWF4X3dvcmRzXV0KICAgICAg"
    "ICAgICAgaWYgbGVuKHdvcmRzKSA+PSAxOgogICAgICAgICAgICAgICAgc2VsZi5fZGF0YS5hcHBlbmQo"
    "KHdvcmRzLCBsYWJlbHMpKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1"
    "cm4gbGVuKHNlbGYuX2RhdGEpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KSAtPiBk"
    "aWN0W3N0ciwgQW55XToKICAgICAgICBjbGVhbl93b3JkcywgY2xlYW5fbGFiZWxzID0gc2VsZi5fZGF0"
    "YVtpZHhdCgogICAgICAgIGlmIG5vdCBzZWxmLmFwcGx5X25vaXNlX2F1ZzoKICAgICAgICAgICAgIyBF"
    "dmFsdWF0aW9uIG1vZGU6IGNsZWFuID09IG5vaXN5LCB0cml2aWFsIDE6MSBhbGlnbm1lbnQKICAgICAg"
    "ICAgICAgYWxpZ25tZW50ID0gWyhbaV0sIFtpXSkgZm9yIGkgaW4gcmFuZ2UobGVuKGNsZWFuX3dvcmRz"
    "KSldCiAgICAgICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICAgICAiY2xlYW5fd29yZHMiIDogY2xl"
    "YW5fd29yZHMsCiAgICAgICAgICAgICAgICAiY2xlYW5fbGFiZWxzIjogbGlzdChjbGVhbl9sYWJlbHMp"
    "LAogICAgICAgICAgICAgICAgIm5vaXN5X3dvcmRzIiA6IGNsZWFuX3dvcmRzLAogICAgICAgICAgICAg"
    "ICAgIm5vaXN5X2xhYmVscyI6IGxpc3QoY2xlYW5fbGFiZWxzKSwKICAgICAgICAgICAgICAgICJhbGln"
    "bm1lbnQiICAgOiBhbGlnbm1lbnQsCiAgICAgICAgICAgIH0KCiAgICAgICAgIyBBcHBseSBjaGFyYWN0"
    "ZXItbGV2ZWwgbm9pc2UgdG8gdGhlIGZ1bGwgc2VudGVuY2Ugc3RyaW5nCiAgICAgICAgY2xlYW5fc3Ry"
    "ID0gIiAiLmpvaW4oY2xlYW5fd29yZHMpCiAgICAgICAgbm9pc3lfc3RyLCBzaGlmdF9sb2csIF8gPSBh"
    "cHBseV9ub2lzZSgKICAgICAgICAgICAgY2xlYW5fc3RyLAogICAgICAgICAgICBwX3N1Yj1OT0lTRV9Q"
    "X1NVQiwKICAgICAgICAgICAgcF9pbnM9Tk9JU0VfUF9JTlMsCiAgICAgICAgICAgIHBfZGVsPU5PSVNF"
    "X1BfREVMLAogICAgICAgICAgICBwX3NwYWNlX2lucz1OT0lTRV9QX1NQQUNFX0lOUywKICAgICAgICAp"
    "CiAgICAgICAgbm9pc3lfd29yZHMgPSBub2lzeV9zdHIuc3BsaXQoKQoKICAgICAgICAjIFdvcmQtbGV2"
    "ZWwgYWxpZ25tZW50IGFuZCBsYWJlbCBwcm9qZWN0aW9uCiAgICAgICAgYWxpZ25tZW50ICAgID0gYnVp"
    "bGRfd29yZF9hbGlnbm1lbnQoY2xlYW5fc3RyLCBzaGlmdF9sb2cpCiAgICAgICAgbm9pc3lfbGFiZWxz"
    "ID0gcHJvamVjdF9sYWJlbHMoY2xlYW5fbGFiZWxzLCBhbGlnbm1lbnQpCgogICAgICAgICMgU2FmZXR5"
    "OiBpZiBhbGlnbm1lbnQgcHJvZHVjZWQgbW9yZS9mZXdlciBub2lzeSBpbmRpY2VzIHRoYW4gYWN0dWFs"
    "IHdvcmRzLAogICAgICAgICMgZmFsbCBiYWNrIHRvIGNsZWFuIChhdm9pZHMgcmFyZSBlZGdlIGNhc2Vz"
    "IGZyb20gdW5leHBlY3RlZCB3aGl0ZXNwYWNlKQogICAgICAgIGV4cGVjdGVkX25vaXN5X2xlbiA9IHN1"
    "bShsZW4obikgZm9yIF8sIG4gaW4gYWxpZ25tZW50KQogICAgICAgIGlmIGV4cGVjdGVkX25vaXN5X2xl"
    "biAhPSBsZW4obm9pc3lfd29yZHMpOgogICAgICAgICAgICBhbGlnbm1lbnQgICAgPSBbKFtpXSwgW2ld"
    "KSBmb3IgaSBpbiByYW5nZShsZW4oY2xlYW5fd29yZHMpKV0KICAgICAgICAgICAgbm9pc3lfd29yZHMg"
    "ID0gY2xlYW5fd29yZHMKICAgICAgICAgICAgbm9pc3lfbGFiZWxzID0gbGlzdChjbGVhbl9sYWJlbHMp"
    "CgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJjbGVhbl93b3JkcyIgOiBjbGVhbl93b3JkcywK"
    "ICAgICAgICAgICAgImNsZWFuX2xhYmVscyI6IGxpc3QoY2xlYW5fbGFiZWxzKSwKICAgICAgICAgICAg"
    "Im5vaXN5X3dvcmRzIiA6IG5vaXN5X3dvcmRzLAogICAgICAgICAgICAibm9pc3lfbGFiZWxzIjogbm9p"
    "c3lfbGFiZWxzLAogICAgICAgICAgICAiYWxpZ25tZW50IiAgIDogYWxpZ25tZW50LAogICAgICAgIH0K"
    "CgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbGxhdGlvbiBmdW5jdGlvbnMKIyAtLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0K"
    "CmRlZiBwcmV0cmFpbl9jb2xsYXRlKGJhdGNoOiBsaXN0W2xpc3Rbc3RyXV0pIC0+IGxpc3RbbGlzdFtz"
    "dHJdXToKICAgICIiIgogICAgQ29sbGF0aW9uIGZvciBQcmV0cmFpbkRhdGFzZXQuCiAgICBNb2RlbHMg"
    "YWNjZXB0IGxpc3RbbGlzdFtzdHJdXSBkaXJlY3RseSDigJQgbm8gdGVuc29yIHBhZGRpbmcgbmVlZGVk"
    "LgogICAgIiIiCiAgICByZXR1cm4gYmF0Y2gKCgpkZWYgbmVyX2NvbGxhdGUoYmF0Y2g6IGxpc3RbZGlj"
    "dFtzdHIsIEFueV1dKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIgogICAgQ29sbGF0aW9uIGZvciBO"
    "RVJEYXRhc2V0LgoKICAgIFBhZHMgbGFiZWwgdGVuc29ycyB3aXRoIE5FUl9JR05PUkVfSU5ERVg7IHBh"
    "c3NlcyB3b3JkIGxpc3RzIHRocm91Z2ggYXMtaXMuCgogICAgUmV0dXJucyBkaWN0IHdpdGgga2V5czoK"
    "ICAgICAgICBjbGVhbl93b3JkcyAgOiBsaXN0W2xpc3Rbc3RyXV0KICAgICAgICBjbGVhbl9sYWJlbHMg"
    "OiAoQiwgTF9jbGVhbl9tYXgpIExvbmdUZW5zb3IKICAgICAgICBub2lzeV93b3JkcyAgOiBsaXN0W2xp"
    "c3Rbc3RyXV0KICAgICAgICBub2lzeV9sYWJlbHMgOiAoQiwgTF9ub2lzeV9tYXgpIExvbmdUZW5zb3IK"
    "ICAgICAgICBhbGlnbm1lbnQgICAgOiBsaXN0W2xpc3RbdHVwbGVbbGlzdFtpbnRdLCBsaXN0W2ludF1d"
    "XV0KICAgICAgICBjbGVhbl9sZW5ndGhzOiBsaXN0W2ludF0KICAgICAgICBub2lzeV9sZW5ndGhzOiBs"
    "aXN0W2ludF0KICAgICIiIgogICAgY2xlYW5fd29yZHMgICA9IFtpdGVtWyJjbGVhbl93b3JkcyJdICBm"
    "b3IgaXRlbSBpbiBiYXRjaF0KICAgIG5vaXN5X3dvcmRzICAgPSBbaXRlbVsibm9pc3lfd29yZHMiXSAg"
    "Zm9yIGl0ZW0gaW4gYmF0Y2hdCiAgICBhbGlnbm1lbnQgICAgID0gW2l0ZW1bImFsaWdubWVudCJdICAg"
    "IGZvciBpdGVtIGluIGJhdGNoXQogICAgY2xlYW5fbGVuZ3RocyA9IFtsZW4odykgZm9yIHcgaW4gY2xl"
    "YW5fd29yZHNdCiAgICBub2lzeV9sZW5ndGhzID0gW2xlbih3KSBmb3IgdyBpbiBub2lzeV93b3Jkc10K"
    "CiAgICBjbGVhbl9sYWJlbHMgPSBfcGFkX2xhYmVsX2JhdGNoKFtpdGVtWyJjbGVhbl9sYWJlbHMiXSBm"
    "b3IgaXRlbSBpbiBiYXRjaF0pCiAgICBub2lzeV9sYWJlbHMgPSBfcGFkX2xhYmVsX2JhdGNoKFtpdGVt"
    "WyJub2lzeV9sYWJlbHMiXSBmb3IgaXRlbSBpbiBiYXRjaF0pCgogICAgcmV0dXJuIHsKICAgICAgICAi"
    "Y2xlYW5fd29yZHMiICA6IGNsZWFuX3dvcmRzLAogICAgICAgICJjbGVhbl9sYWJlbHMiIDogY2xlYW5f"
    "bGFiZWxzLAogICAgICAgICJub2lzeV93b3JkcyIgIDogbm9pc3lfd29yZHMsCiAgICAgICAgIm5vaXN5"
    "X2xhYmVscyIgOiBub2lzeV9sYWJlbHMsCiAgICAgICAgImFsaWdubWVudCIgICAgOiBhbGlnbm1lbnQs"
    "CiAgICAgICAgImNsZWFuX2xlbmd0aHMiOiBjbGVhbl9sZW5ndGhzLAogICAgICAgICJub2lzeV9sZW5n"
    "dGhzIjogbm9pc3lfbGVuZ3RocywKICAgIH0K"
)
with open("/kaggle/working/src/data/dataset.py", "wb") as _f:
    _f.write(_b64.decodebytes(_D.encode()))
print("dataset.py: written from embedded source")

In [ ]:
# ── Patch stale config.py before importing ───────────────────────────────────
# Fixes uploaded dataset having old dataset name (conll2003 -> eriktks/conll2003)
p = '/kaggle/working/src/config.py'
with open(p) as f:
    txt = f.read()
txt = txt.replace('"conll2003"', '"eriktks/conll2003"')
with open(p, 'w') as f:
    f.write(txt)
print('config.py patched — conll dataset:', end=" ")
[print(l.strip()) for l in txt.splitlines() if "conll2003" in l and "DATASET" in l]

In [ ]:
# â”€â”€ 3. Verify GPU â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# â”€â”€ 4. Verify imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/kaggle/working')

from src.config import DEVICE, RETVEC_WEIGHTS, WEIGHTS_DIR, STAGE1_CKPT
from src.models.retbert import RetBERT
from src.data.dataset import PretrainDataset, pretrain_collate
from src.losses import stage1_loss

print('Device          :', DEVICE)
print('RetVec weights  :', RETVEC_WEIGHTS, '| exists:', RETVEC_WEIGHTS.exists())
print('Checkpoint path :', STAGE1_CKPT)

# Quick forward-pass check with a tiny batch
model = RetBERT().to(DEVICE)
with torch.no_grad():
    _, pooled = model([['hello', 'world'], ['test']])
print('RetBERT forward OK â€” pooled shape:', tuple(pooled.shape))
del model

In [ ]:
# â”€â”€ 5. Hyperparameters (override config defaults here if needed) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import src.config as cfg

# Uncomment to reduce epochs for a quick test run
# cfg.STAGE1_EPOCHS     = 2
# cfg.STAGE1_BATCH_SIZE = 16   # lower if OOM

print(f'Epochs      : {cfg.STAGE1_EPOCHS}')
print(f'Batch size  : {cfg.STAGE1_BATCH_SIZE}')
print(f'LR          : {cfg.STAGE1_LR}')
print(f'Warmup steps: {cfg.STAGE1_WARMUP_STEPS}')
print(f'Max words   : {cfg.STAGE1_MAX_WORDS}')

In [ ]:
# â”€â”€ 6. Load dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# WikiText-103 + CoNLL-2003 train split (~1.2M sentences after filtering)
from src.data.dataset import PretrainDataset

train_dataset = PretrainDataset(split='train', max_words=cfg.STAGE1_MAX_WORDS, verbose=True)

In [ ]:
# â”€â”€ 7. Build models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import math
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# BERT teacher (frozen)
print('Loading BERT teacher ...')
bert_tokenizer = AutoTokenizer.from_pretrained(cfg.BERT_MODEL_NAME)
bert = AutoModel.from_pretrained(cfg.BERT_MODEL_NAME)
bert.eval()
for p in bert.parameters():
    p.requires_grad_(False)
bert = bert.to(DEVICE)
print(f'  BERT params: {sum(p.numel() for p in bert.parameters()):,}  (all frozen)')

# RetBERT student
retbert = RetBERT().to(DEVICE)
trainable = [p for p in retbert.parameters() if p.requires_grad]
print(f'  RetBERT trainable: {sum(p.numel() for p in trainable):,}')

In [ ]:
# â”€â”€ 8. Training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@torch.no_grad()
def bert_sentence_emb(batch_words):
    texts = [' '.join(w) for w in batch_words]
    enc = bert_tokenizer(
        texts, max_length=cfg.STAGE1_MAX_SUBWORDS,
        truncation=True, padding=True, return_tensors='pt'
    ).to(DEVICE)
    out = bert(**enc)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    return (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def make_scheduler(optimizer, warmup, total):
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        p = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(optimizer, lr_lambda)

loader = DataLoader(
    train_dataset,
    batch_size=cfg.STAGE1_BATCH_SIZE,
    shuffle=True,
    collate_fn=pretrain_collate,
    num_workers=2,
    pin_memory=True,
)

optimizer   = AdamW(trainable, lr=cfg.STAGE1_LR, weight_decay=0.01)
total_steps = cfg.STAGE1_EPOCHS * len(loader)
scheduler   = make_scheduler(optimizer, cfg.STAGE1_WARMUP_STEPS, total_steps)

loss_history = []
best_loss    = float('inf')

for epoch in range(1, cfg.STAGE1_EPOCHS + 1):
    retbert.train()
    epoch_loss = 0.0

    for step, batch_words in enumerate(loader, 1):
        z_bert    = bert_sentence_emb(batch_words)
        _, z_rb   = retbert(batch_words)
        loss      = stage1_loss(z_bert, z_rb)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        if step % 500 == 0:
            avg = epoch_loss / step
            lr  = scheduler.get_last_lr()[0]
            print(f'  E{epoch} {step}/{len(loader)}  loss={avg:.4f}  lr={lr:.2e}', flush=True)

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f'Epoch {epoch}/{cfg.STAGE1_EPOCHS}  avg_loss={avg:.4f}')

    if avg < best_loss:
        best_loss = avg
        torch.save(retbert.state_dict(), str(cfg.STAGE1_CKPT))
        print(f'  -> saved {cfg.STAGE1_CKPT}')

print(f'\nDone. Best loss: {best_loss:.4f}')

In [ ]:
# â”€â”€ 9. Loss curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Avg cosine distance loss')
plt.title('Stage 1 training loss')
plt.tight_layout()
plt.savefig('/kaggle/working/stage1_loss.png', dpi=100)
plt.show()

import os
ckpt = cfg.STAGE1_CKPT
print(f'Checkpoint: {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB)')
print('Download this file and upload as a Kaggle dataset for Stage 2.')